## Load Data

In [2]:
!ls -lh /content

total 157M
-rw-r--r-- 1 root root 157M Dec 12 18:09 favorita_model_ready.parquet
drwxr-xr-x 1 root root 4.0K Dec 11 14:34 sample_data


In [1]:
import pandas as pd

df = pd.read_parquet("/content/favorita_model_ready_2013_2015.parquet")

In [2]:
df.shape


(13913510, 27)

In [3]:
df.columns


Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion',
       'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster',
       'dcoilwtico', 'description', 'transactions', 'year', 'month',
       'dayofweek', 'weekofyear', 'is_weekend', 'is_holiday', 'lag_7',
       'lag_14', 'lag_28', 'rolling_7', 'rolling_14'],
      dtype='object')

In [4]:
df.tail()


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,month,dayofweek,weekofyear,is_weekend,is_holiday,lag_7,lag_14,lag_28,rolling_7,rolling_14
13913505,66368775,2015-12-31,4,314570,6.0,0,CLEANING,3024,0,Quito,...,12,3,53,0,1,9.0,5.0,3.0,6.142857,6.500000
13913506,66442398,2015-12-31,46,841514,3.0,0,GROCERY I,1072,0,Quito,...,12,3,53,0,1,10.0,7.0,7.0,15.142857,13.428571
13913507,66427101,2015-12-31,38,1146802,8.0,0,GROCERY I,1040,0,Loja,...,12,3,53,0,1,3.0,7.0,3.0,4.000000,5.285714
13913508,66368777,2015-12-31,4,315176,8.0,0,BEVERAGES,1124,0,Quito,...,12,3,53,0,1,4.0,4.0,21.0,10.142857,12.642857
13913509,66455563,2015-12-31,51,2026983,16.0,0,GROCERY I,1034,0,Guayaquil,...,12,3,53,0,1,31.0,28.0,18.0,24.285714,52.357143


In [5]:
df['date'].dtype

dtype('<M8[ns]')

In [22]:
FEATURES = [
    # identifiers
    "store_nbr",
    "item_nbr",

    # categorical structure
    "family",
    "class",
    "city",
    "cluster",

    # promotions & ops
    "onpromotion",
    "transactions",
    "perishable",

    # calendar
    "month",
    "dayofweek",
    "weekofyear",
    "is_weekend",

    # external signals
    "dcoilwtico",
    "is_holiday",

    # time-series features
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_7",
    "rolling_14"
]

CATEGORICAL_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "cluster"
]

for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")

TARGET = "unit_sales"

## Time-aware Train/Test Split

In [23]:
cutoff_date = "2017-06-01"

train_df = df[df["date"] < cutoff_date]
valid_df = df[df["date"] >= cutoff_date]

len(train_df), len(valid_df)


(6023898, 397009)

## Define features and target

In [8]:
df.columns

Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion',
       'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster',
       'dcoilwtico', 'description', 'transactions', 'year', 'month',
       'dayofweek', 'weekofyear', 'is_weekend', 'is_holiday', 'lag_7',
       'lag_14', 'lag_28', 'rolling_7', 'rolling_14'],
      dtype='object')

In [24]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_valid = valid_df[FEATURES]
y_valid = valid_df[TARGET]

In [25]:
print("Train rows:", X_train.shape[0])
print("Valid rows:", X_valid.shape[0])

print("\nCategorical dtypes:")
print(X_train[CATEGORICAL_FEATURES].dtypes)


Train rows: 6023898
Valid rows: 397009

Categorical dtypes:
store_nbr    category
item_nbr     category
family       category
class        category
city         category
cluster      category
dtype: object


In [26]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse"
)



[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.070604 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2571
[LightGBM] [Info] Number of data points in the train set: 6023898, number of used features: 20
[LightGBM] [Info] Start training from score 30.225520


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
              n_jobs=-1, num_leaves=64, random_state=42, subsample=0.8)

In [27]:
import numpy as np
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

rmse


np.float64(27.81019894933662)

In [29]:
baseline_pred = X_valid["lag_7"]

baseline_rmse = np.sqrt(
    mean_squared_error(y_valid, baseline_pred)
)

baseline_rmse


np.float64(40.005072406250434)

In [30]:
baseline_pred_roll = X_valid["rolling_7"]

baseline_rmse_roll = np.sqrt(
    mean_squared_error(y_valid, baseline_pred_roll)
)

baseline_rmse_roll


np.float64(32.21615596576178)

In [32]:
from sklearn.metrics import mean_squared_log_error

y_pred = pd.Series(y_pred, index=y_valid.index)

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid.clip(lower=0),
        y_pred.clip(lower=0)
    )
)

rmsle

np.float64(0.5330477146416599)

## Log-target LightGBM training + RMSLE evaluation

In [33]:
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [34]:
model_log = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_log.fit(
    X_train,
    y_train_log,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid_log)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50)
    ]
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.477590 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2571
[LightGBM] [Info] Number of data points in the train set: 6023898, number of used features: 20
[LightGBM] [Info] Start training from score -680622392496873871147351714824192.000000
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 2.7489e+35	valid_0's l2: 7.55645e+70


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
              n_jobs=-1, num_leaves=64, random_state=42, subsample=0.8)

In [39]:
y_pred_log = model_log.predict(
    X_valid,
    num_iteration=model_log.best_iteration_
)

y_pred = np.expm1(y_pred_log)



In [40]:
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid.clip(lower=0),
        np.clip(y_pred, 0, None)
    )
)

rmse, rmsle

(np.float64(61.37445519554005), np.float64(3.0354103239073713))

In [41]:
y_valid.describe()


,unit_sales
count,397009.000000
mean,29.944653
std,53.002446
min,-274.000000
25%,8.000000
50%,16.000000
75%,33.000000
max,6932.000000


In [42]:
y_valid.head()


,unit_sales
6023898,23.0
6023899,3.0
6023900,44.0
6023901,4.0
6023902,6.0


In [43]:
np.allclose(y_valid, np.log1p(np.expm1(y_valid)))


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in expm1
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


False

In [44]:
y_train_raw = train_df["unit_sales"].values
y_valid_raw = valid_df["unit_sales"].values

In [45]:
y_train_raw = np.clip(y_train_raw, 0, None)
y_valid_raw = np.clip(y_valid_raw, 0, None)

In [46]:
y_train_log = np.log1p(y_train_raw)
y_valid_log = np.log1p(y_valid_raw)

In [47]:
model_log = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_log.fit(
    X_train,
    y_train_log,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid_log)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50)
    ]
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.685559 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2571
[LightGBM] [Info] Number of data points in the train set: 6023898, number of used features: 20
[LightGBM] [Info] Start training from score 2.910365
Training until validation scores don't improve for 30 rounds
[50]	valid_0's rmse: 0.517537	valid_0's l2: 0.267845
[100]	valid_0's rmse: 0.502294	valid_0's l2: 0.252299
[150]	valid_0's rmse: 0.497813	valid_0's l2: 0.247818
[200]	valid_0's rmse: 0.495501	valid_0's l2: 0.245521
[250]	valid_0's rmse: 0.493939	valid_0's l2: 0.243976
[300]	valid_0's rmse: 0.49306	valid_0's l2: 0.243108
[350]	valid_0's rmse: 0.492301	valid_0's l

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
              n_jobs=-1, num_leaves=64, random_state=42, subsample=0.8)

In [48]:
# Predict in log space using best iteration
y_pred_log = model_log.predict(
    X_valid,
    num_iteration=model_log.best_iteration_
)

# Invert log transform
y_pred = np.expm1(y_pred_log)


In [49]:
from sklearn.metrics import mean_squared_error, mean_squared_log_error
import numpy as np

# RMSE (original scale)
rmse_log_model = np.sqrt(
    mean_squared_error(y_valid_raw, y_pred)
)

# RMSLE (primary metric)
rmsle_log_model = np.sqrt(
    mean_squared_log_error(
        y_valid_raw,
        np.clip(y_pred, 0, None)
    )
)

rmse_log_model, rmsle_log_model


(np.float64(28.47586464861986), np.float64(0.49087099129589895))

## Optuna + ML FLow Hyperparameter Tuning

In [50]:
!pip install mlflow
!pip install optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 7.2 MB/s eta 0:00:00


In [51]:
import optuna
import mlflow
import mlflow.lightgbm

def objective_lgb(trial):
    params = {
        "n_estimators": 500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.7, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.7, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }

    with mlflow.start_run(nested=True):
        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train,
            y_train_log,
            categorical_feature=CATEGORICAL_FEATURES,
            eval_set=[(X_valid, y_valid_log)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(30)]
        )

        y_pred_log = model.predict(
            X_valid,
            num_iteration=model.best_iteration_
        )
        y_pred = np.expm1(y_pred_log)

        rmsle = np.sqrt(
            mean_squared_log_error(
                y_valid_raw,
                np.clip(y_pred, 0, None)
            )
        )

        mlflow.log_params(params)
        mlflow.log_metric("rmsle", rmsle)

        return rmsle


In [52]:
study = optuna.create_study(direction="minimize")
study.optimize(objective_lgb, n_trials=25)


[I 2025-12-12 19:15:28,058] A new study created in memory with name: no-name-fc2af93b-03cc-4d5e-a616-fee8fe90ecac
2025/12/12 19:15:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/12 19:15:34 INFO mlflow.store.db.utils: Updating database tables
2025/12/12 19:15:34 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/12 19:15:34 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/12 19:15:34 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/12 19:15:34 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/12 19:15:34 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/12 19:15:34 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/12 19:15:34 INFO alembic.runtime.migration: Running upgrade df50e92ffc

[LightGBM] [Warning] min_data_in_leaf is set=155, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=155
[LightGBM] [Warning] feature_fraction is set=0.8972879877179761, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8972879877179761
[LightGBM] [Warning] bagging_fraction is set=0.9583256115739262, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9583256115739262
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=155, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=155
[LightGBM] [Warning] feature_fraction is set=0.8972879877179761, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8972879877179761
[LightGBM] [Warning] bagging_fraction is set=0.9583256115739262, s

[I 2025-12-12 19:22:40,859] Trial 0 finished with value: 0.4916474211101287 and parameters: {'learning_rate': 0.06264692393663822, 'num_leaves': 42, 'min_data_in_leaf': 155, 'feature_fraction': 0.8972879877179761, 'bagging_fraction': 0.9583256115739262}. Best is trial 0 with value: 0.4916474211101287.


[LightGBM] [Warning] min_data_in_leaf is set=160, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=160
[LightGBM] [Warning] feature_fraction is set=0.9465922702389011, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9465922702389011
[LightGBM] [Warning] bagging_fraction is set=0.8118665658361347, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8118665658361347
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=160, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=160
[LightGBM] [Warning] feature_fraction is set=0.9465922702389011, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9465922702389011
[LightGBM] [Warning] bagging_fraction is set=0.8118665658361347, s

[I 2025-12-12 19:33:27,936] Trial 1 finished with value: 0.48893979834312773 and parameters: {'learning_rate': 0.057840792030508056, 'num_leaves': 112, 'min_data_in_leaf': 160, 'feature_fraction': 0.9465922702389011, 'bagging_fraction': 0.8118665658361347}. Best is trial 1 with value: 0.48893979834312773.


...

[I 2025-12-12 22:39:19,786] Trial 22 finished with value: 0.48864469018855555 and parameters: {'learning_rate': 0.07844401581042017, 'num_leaves': 127, 'min_data_in_leaf': 107, 'feature_fraction': 0.7963609736931693, 'bagging_fraction': 0.7680037709221788}. Best is trial 8 with value: 0.48811331852399.


[LightGBM] [Warning] min_data_in_leaf is set=77, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=77
[LightGBM] [Warning] feature_fraction is set=0.8065856576712195, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8065856576712195
[LightGBM] [Warning] bagging_fraction is set=0.7185621601970078, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7185621601970078
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=77, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=77
[LightGBM] [Warning] feature_fraction is set=0.8065856576712195, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8065856576712195
[LightGBM] [Warning] bagging_fraction is set=0.7185621601970078, subsa

[I 2025-12-12 22:46:53,045] Trial 23 finished with value: 0.4884729080404506 and parameters: {'learning_rate': 0.09192306908858203, 'num_leaves': 116, 'min_data_in_leaf': 77, 'feature_fraction': 0.8065856576712195, 'bagging_fraction': 0.7185621601970078}. Best is trial 8 with value: 0.48811331852399.


[LightGBM] [Warning] min_data_in_leaf is set=144, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=144
[LightGBM] [Warning] feature_fraction is set=0.8591273225296187, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8591273225296187
[LightGBM] [Warning] bagging_fraction is set=0.8320213227405258, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8320213227405258
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=144, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=144
[LightGBM] [Warning] feature_fraction is set=0.8591273225296187, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8591273225296187
[LightGBM] [Warning] bagging_fraction is set=0.8320213227405258, s

[I 2025-12-12 22:56:00,821] Trial 24 finished with value: 0.48885966312000734 and parameters: {'learning_rate': 0.07730947002773758, 'num_leaves': 120, 'min_data_in_leaf': 144, 'feature_fraction': 0.8591273225296187, 'bagging_fraction': 0.8320213227405258}. Best is trial 8 with value: 0.48811331852399.


In [53]:
import optuna
import joblib

# Save study to disk
joblib.dump(study, "optuna_lgb_study.pkl")


['optuna_lgb_study.pkl']

In [54]:
from google.colab import files
files.download("optuna_lgb_study.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
best_params = study.best_trial.params
best_score = study.best_value

print("Best RMSLE:", best_score)
print("Best params:", best_params)


Best RMSLE: 0.48811331852399
Best params: {'learning_rate': 0.09556996173730738, 'num_leaves': 120, 'min_data_in_leaf': 143, 'feature_fraction': 0.8889602666162978, 'bagging_fraction': 0.723716991047944}


In [57]:
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import numpy as np
from sklearn.metrics import mean_squared_log_error

with mlflow.start_run(run_name="final_lgb_model"):

    final_model = lgb.LGBMRegressor(
        n_estimators=800,
        random_state=42,
        n_jobs=-1,
        **best_params
    )

    final_model.fit(
        X_train,
        y_train_log,
        categorical_feature=CATEGORICAL_FEATURES,
        eval_set=[(X_valid, y_valid_log)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )

    # Predict & invert log
    y_pred_log = final_model.predict(X_valid)
    y_pred = np.expm1(y_pred_log)

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_valid_raw,
            np.clip(y_pred, 0, None)
        )
    )

    mlflow.log_params(best_params)
    mlflow.log_metric("rmsle", rmsle)

    mlflow.lightgbm.log_model(
        final_model,
        artifact_path="model",
        registered_model_name="favorita_lgb"
    )

print("Final RMSLE:", rmsle)


[LightGBM] [Warning] min_data_in_leaf is set=143, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=143
[LightGBM] [Warning] feature_fraction is set=0.8889602666162978, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8889602666162978
[LightGBM] [Warning] bagging_fraction is set=0.723716991047944, subsample=1.0 will be ignored. Current value: bagging_fraction=0.723716991047944
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=143, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=143
[LightGBM] [Warning] feature_fraction is set=0.8889602666162978, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8889602666162978
[LightGBM] [Warning] bagging_fraction is set=0.723716991047944, subs

2025/12/12 23:25:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/12 23:25:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/12 23:25:20 INFO mlflow.store.db.utils: Updating database tables
2025/12/12 23:25:20 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/12 23:25:20 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Final RMSLE: 0.48808458307225616


Successfully registered model 'favorita_lgb'.
Created version '1' of model 'favorita_lgb'.


In [58]:
final_model.booster_.save_model("favorita_lgb.txt")
files.download("favorita_lgb.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
import shutil

shutil.make_archive("mlflow_run", "zip", "mlruns")
files.download("mlflow_run.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
# Prepare test data
y_test_raw = np.clip(test_df["unit_sales"].values, 0, None)
X_test = test_df[FEATURES]

# Predict
y_test_log_pred = final_model.predict(X_test)
y_test_pred = np.expm1(y_test_log_pred)

test_rmsle = np.sqrt(
    mean_squared_log_error(y_test_raw, np.clip(y_test_pred, 0, None))
)

print("Test RMSLE:", test_rmsle)

mlflow.log_metric("test_rmsle", test_rmsle)


NameError: name 'test_df' is not defined